In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
sys.path.append('../')
import utils.utilfunc as ut
import Jobcontrol as jc
import datetime

In [0]:

#Job Parameters
rundate = ut.get_rundate()
schema_name = 'warehouse.edw'
table_name = 'dim_date'
table_full_name = f"{schema_name}.{table_name}"
staging_table = "warehouse.edw_stg.dim_date_stg"
print("Job Triggered for rundate: ",rundate)


In [0]:
#Get Max Timestamp from Job_control Table
from pyspark.sql.functions import col,to_timestamp
max_timestamp = jc.get_max_timestamp(spark,schema_name,table_name)
print("Max Timestamp from Job Control Table: ",max_timestamp)

In [0]:
import datetime as dt
timeval = dt.datetime.strptime(max_timestamp,"%Y-%m-%d %H:%M:%S.%f")
print(timeval)

In [0]:
from pyspark.sql.functions import col,to_timestamp,expr,date_format
df=spark.read.table(staging_table)
df.printSchema()
dfgld = df.withColumn('row_wid',date_format(col('date'),"yyyyMMdd")).select(['row_wid','date','day','month','year','dayofweek','insert_dt','rundate','update_dt'])

dfgld = dfgld.withColumnRenamed('dayofweek','day_of_week')
#dfgld.show()
#dffil.show()

In [0]:
#Scd1 Loading
from delta.tables import DeltaTable
dc = DeltaTable.forName(spark, table_full_name)

display(dc) 
dc.alias('dim_date').merge(dfgld.alias('dim_stg'), 'dim_stg.date=dim_date.date').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()



In [0]:
%sql
select * from warehouse.edw.dim_date

In [0]:
#Update Job Control Table
jc.insert_log(spark,schema_name,table_name,max_timestamp,rundate)

In [0]:
%sql
select * from warehouse.edw.job_control where schema_name = 'warehouse.edw' and table_name = 'dim_date'

In [0]:
%sql
select * from warehouse.edw_stg.dim_date_stg